# 08 · Ajuste fino QLoRA de Gemma-2 (Transformers + PEFT)

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca  
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

Misma configuración de ajuste fino que el notebook 07, para Gemma-2 2B y 9B, usando Transformers + PEFT con atención eager. Esta ejecución corresponde a la sesión final de Gemma-2 2B, el modelo EmoGol-2B.

*Nombre durante el desarrollo: `06_entrenamiento_real_gemma_sin_unsloth` (algunas salidas conservan esa numeración).*

## 1. Instalar dependencias

In [1]:
!pip install -U transformers accelerate peft bitsandbytes trl huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 27.2 MB/s eta 0:00:00


## 2. Configuración

In [2]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

TU_USUARIO_HF = "santiagortegasegura"

MODELOS_DISPONIBLES = {
    "gemma2_2b": "google/gemma-2-2b-it",
    "gemma2_9b": "google/gemma-2-9b-it",
}
CLAVE_MODELO = "gemma2_2b"
MODEL_ID = MODELOS_DISPONIBLES[CLAVE_MODELO]

REPO_DATOS = f"{TU_USUARIO_HF}/emotion-futbol-data"
REPO_CHECKPOINTS = f"{TU_USUARIO_HF}/emotion-futbol-checkpoints"

COLUMNA_TEXTO = "text"
MAX_LENGTH = 112
EPOCAS = 3

print(f"Modelo a entrenar: {CLAVE_MODELO} -> {MODEL_ID}")

Modelo a entrenar: gemma2_2b -> google/gemma-2-2b-it


## 3. Revisar si el modelo ya está entrenado

In [3]:
from huggingface_hub import hf_hub_download

YA_COMPLETO = True
try:
    hf_hub_download(repo_id=REPO_CHECKPOINTS, repo_type="dataset",
                     filename=f"{CLAVE_MODELO}/ENTRENAMIENTO_COMPLETO.txt")
except Exception:
    YA_COMPLETO = False

if YA_COMPLETO:
    raise SystemExit(f"'{CLAVE_MODELO}' ya está completamente entrenado (ENTRENAMIENTO_COMPLETO.txt "
                      f"ya existe en tu bodega). No hace falta correr nada más para este modelo.")
else:
    print(f"'{CLAVE_MODELO}' todavía no está completo -- seguimos.")

'gemma2_2b' todavía no está completo -- seguimos.


## 4. Buscar un checkpoint previo

In [4]:
import glob
from huggingface_hub import snapshot_download

LOCAL_CKPT_DIR = f"/content/ckpt_{CLAVE_MODELO}"

try:
    snapshot_download(
        repo_id=REPO_CHECKPOINTS, repo_type="dataset",
        allow_patterns=f"{CLAVE_MODELO}/checkpoint-*/*",
        local_dir=LOCAL_CKPT_DIR,
    )
except Exception as e:
    print("No se encontraron checkpoints previos (o aún no existen):", e)

candidatos = glob.glob(f"{LOCAL_CKPT_DIR}/{CLAVE_MODELO}/checkpoint-*")
RESUME_FROM = None
if candidatos:
    candidatos.sort(key=lambda p: int(p.rstrip("/").split("-")[-1]))
    RESUME_FROM = candidatos[-1]
    print(f"Se encontró un checkpoint previo -- retomando desde: {RESUME_FROM}")
else:
    print("No hay checkpoints previos -- este modelo empieza desde cero.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Se encontró un checkpoint previo -- retomando desde: /content/ckpt_gemma2_2b/gemma2_2b/checkpoint-13500


## 5. Descargar los datos

In [5]:
import pandas as pd

ruta_train = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset",
                              filename="04b_entrenamiento_72k.csv")
ruta_val_interna = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset",
                                    filename="04b_validacion_interna_18k.csv")

df_entrenamiento = pd.read_csv(ruta_train)
df_validacion_interna = pd.read_csv(ruta_val_interna)

print(f"Entrenamiento: {len(df_entrenamiento)} ejemplos")
print(f"Validación interna: {len(df_validacion_interna)} ejemplos")

04b_entrenamiento_72k.csv:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

04b_validacion_interna_18k.csv:   0%|          | 0.00/2.16M [00:00<?, ?B/s]

Entrenamiento: 72000 ejemplos
Validación interna: 18000 ejemplos


## 6. Cargar el modelo en 4-bit CON atención "eager" real (sin Unsloth)

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

torch.cuda.reset_peak_memory_stats()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Modelo y LoRA cargados sin errores (atención eager confirmada).")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881
Modelo y LoRA cargados sin errores (atención eager confirmada).


## 7. Preparar los datasets con el formato de prompt

In [7]:
EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']

def formatear_prompt(row):
    return (
        f"Clasifica la emoción del siguiente comentario de fútbol en una de estas categorías: "
        f"{', '.join(EMOCIONES)}.\n\nComentario: {row[COLUMNA_TEXTO]}\n\nEmoción: {row['label']}"
    )

df_entrenamiento['texto_formateado'] = df_entrenamiento.apply(formatear_prompt, axis=1)

df_validacion_interna['texto_formateado'] = df_validacion_interna.apply(formatear_prompt, axis=1)

from datasets import Dataset
dataset_entrenamiento = Dataset.from_pandas(df_entrenamiento[['texto_formateado']])
dataset_eval = Dataset.from_pandas(df_validacion_interna[['texto_formateado']])

print(f"Entrenamiento: {len(dataset_entrenamiento)} ejemplos | Evaluación (18k completas): {len(dataset_eval)} ejemplos")

Entrenamiento: 72000 ejemplos | Evaluación (18k completas): 18000 ejemplos


## 8. Entrenar (con checkpoints en Hugging Face)

In [8]:
import time
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from huggingface_hub import upload_folder, HfApi

api_hf = HfApi()

MINUTOS_ENTRE_GUARDADOS = 20

class GuardarPorTiempoCallback(TrainerCallback):
    def __init__(self, minutos=MINUTOS_ENTRE_GUARDADOS):
        self.intervalo_seg = minutos * 60
        self.ultimo_guardado = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        ahora = time.time()
        if control.should_save:
            self.ultimo_guardado = ahora
        elif ahora - self.ultimo_guardado >= self.intervalo_seg:
            control.should_save = True
            self.ultimo_guardado = ahora
        return control

SUBIDAS_ENTRE_SQUASH = 8
class SubirCheckpointCallback(TrainerCallback):
    def __init__(self):
        self.subidas_desde_squash = 0

    def on_save(self, args, state, control, **kwargs):
        paso_actual = state.global_step
        ckpt_dir = f"{args.output_dir}/checkpoint-{paso_actual}"
        try:
            upload_folder(
                folder_path=ckpt_dir,
                repo_id=REPO_CHECKPOINTS,
                repo_type="dataset",
                path_in_repo=f"{CLAVE_MODELO}/checkpoint-{paso_actual}",
            )
            print(f"  -> Checkpoint del paso {paso_actual} subido a Hugging Face.")
        except Exception as e:
            print(f"  -> No se pudo subir el checkpoint del paso {paso_actual} "
                  f"(sigue entrenando de todas formas): {e}")
            return

        try:
            archivos = api_hf.list_repo_files(repo_id=REPO_CHECKPOINTS, repo_type="dataset")
            prefijo = f"{CLAVE_MODELO}/checkpoint-"
            pasos_viejos = {
                int(a[len(prefijo):].split("/")[0])
                for a in archivos
                if a.startswith(prefijo) and a[len(prefijo):].split("/")[0].isdigit()
                and int(a[len(prefijo):].split("/")[0]) != paso_actual
            }
            for paso_viejo in pasos_viejos:
                api_hf.delete_folder(repo_id=REPO_CHECKPOINTS, repo_type="dataset",
                                      path_in_repo=f"{CLAVE_MODELO}/checkpoint-{paso_viejo}")
                print(f"  -> Checkpoint viejo del paso {paso_viejo} borrado de Hugging Face "
                      f"(para no acumular espacio ni hacer más lento el próximo 'retomar').")
        except Exception as e:
            print(f"  -> No se pudo limpiar checkpoints viejos (no es grave, solo ocupa más espacio): {e}")

        self.subidas_desde_squash += 1
        if self.subidas_desde_squash >= SUBIDAS_ENTRE_SQUASH:
            try:
                api_hf.super_squash_history(repo_id=REPO_CHECKPOINTS, repo_type="dataset")
                print("  -> Historial de la bodega comprimido (libera el espacio de los "
                      "checkpoints viejos de verdad, no solo del árbol actual).")
                self.subidas_desde_squash = 0
            except Exception as e:
                print(f"  -> No se pudo comprimir el historial todavía (no es grave, se "
                      f"reintenta en el próximo guardado): {e}")

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset_entrenamiento,
    eval_dataset=dataset_eval,
    args=SFTConfig(
        dataset_text_field="texto_formateado",
        max_length=MAX_LENGTH,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=EPOCAS,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy="epoch",
        save_strategy="steps",
        save_steps=150,
        save_total_limit=3,
        output_dir=f"{LOCAL_CKPT_DIR}/{CLAVE_MODELO}",
        gradient_checkpointing=True,
    ),
)
trainer.add_callback(GuardarPorTiempoCallback())
trainer.add_callback(SubirCheckpointCallback())

trainer.train(resume_from_checkpoint=RESUME_FROM)

print(f"\nPico de VRAM usada durante todo el entrenamiento: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Adding EOS to train dataset:   0%|          | 0/72000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/72000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/72000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/72000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/72000 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/18000 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/18000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Epoch,Training Loss,Validation Loss



Pico de VRAM usada durante todo el entrenamiento: 4.59 GB


## 9. Guardar y subir el adaptador final

In [9]:
from huggingface_hub import upload_file

CARPETA_FINAL = f"{LOCAL_CKPT_DIR}/{CLAVE_MODELO}/adapter_final"
model.save_pretrained(CARPETA_FINAL)
tokenizer.save_pretrained(CARPETA_FINAL)

upload_folder(
    folder_path=CARPETA_FINAL,
    repo_id=REPO_CHECKPOINTS,
    repo_type="dataset",
    path_in_repo=f"{CLAVE_MODELO}/adapter_final",
)

with open("/content/marca_completo.txt", "w") as f:
    f.write(f"Entrenamiento completo: {CLAVE_MODELO} ({MODEL_ID}), {EPOCAS} épocas sobre 72k, sin Unsloth.\n")

upload_file(
    path_or_fileobj="/content/marca_completo.txt",
    path_in_repo=f"{CLAVE_MODELO}/ENTRENAMIENTO_COMPLETO.txt",
    repo_id=REPO_CHECKPOINTS,
    repo_type="dataset",
)

try:
    archivos = api_hf.list_repo_files(repo_id=REPO_CHECKPOINTS, repo_type="dataset")
    prefijo = f"{CLAVE_MODELO}/checkpoint-"
    pasos_a_borrar = {a[len(prefijo):].split("/")[0] for a in archivos if a.startswith(prefijo)}
    for paso in pasos_a_borrar:
        api_hf.delete_folder(repo_id=REPO_CHECKPOINTS, repo_type="dataset",
                              path_in_repo=f"{CLAVE_MODELO}/checkpoint-{paso}")
    if pasos_a_borrar:
        print(f"Checkpoints intermedios de '{CLAVE_MODELO}' borrados (ya no hacen falta, quedó el adapter_final).")
except Exception as e:
    print(f"No se pudieron borrar los checkpoints intermedios (no es grave): {e}")

try:
    api_hf.super_squash_history(repo_id=REPO_CHECKPOINTS, repo_type="dataset")
    print("Historial de la bodega comprimido -- el espacio de los checkpoints intermedios ya quedó liberado de verdad.")
except Exception as e:
    print(f"No se pudo comprimir el historial ahora (no es grave, puedes correr esto luego a mano): {e}")

print(f"\nListo -- '{CLAVE_MODELO}' entrenado y guardado en tu bodega de Hugging Face.")

Checkpoints intermedios de 'gemma2_2b' borrados (ya no hacen falta, quedó el adapter_final).
Historial de la bodega comprimido -- el espacio de los checkpoints intermedios ya quedó liberado de verdad.

Listo -- 'gemma2_2b' entrenado y guardado en tu bodega de Hugging Face.


In [11]:
from huggingface_hub import list_repo_files, hf_hub_download
import re, json

REPO = "santiagortegasegura/emotion-futbol-checkpoints"
TOTAL_STEPS = 13500

files = list_repo_files(REPO, repo_type="dataset")

modelos = {}
for f in files:
    m = re.match(r"([^/]+)/checkpoint-(\d+)/", f)
    if m:
        clave, paso = m.group(1), int(m.group(2))
        modelos[clave] = max(modelos.get(clave, 0), paso)

print(f"{'Modelo':<15}{'Último checkpoint':<20}{'% completado'}")
for clave, paso in sorted(modelos.items()):
    pct = paso / TOTAL_STEPS * 100
    estado = "COMPLETO" if paso >= TOTAL_STEPS else "en curso"
    print(f"{clave:<15}{paso:<20}{pct:.1f}% ({estado})")

Modelo         Último checkpoint   % completado
gemma2_9b      12027               89.1% (en curso)


In [12]:
from huggingface_hub import login, list_repo_files, hf_hub_download
import re, json

login()

REPO = "santiagortegasegura/emotion-futbol-checkpoints"
TOTAL_STEPS = 13500

files = list_repo_files(REPO, repo_type="dataset")

modelos = {}
for f in files:
    m = re.match(r"([^/]+)/checkpoint-(\d+)/", f)
    if m:
        clave, paso = m.group(1), int(m.group(2))
        modelos[clave] = max(modelos.get(clave, 0), paso)

print(f"{'Modelo':<15}{'Último checkpoint':<20}{'% completado'}")
for clave, paso in sorted(modelos.items()):
    pct = paso / TOTAL_STEPS * 100
    estado = "COMPLETO" if paso >= TOTAL_STEPS else "en curso"
    print(f"{clave:<15}{paso:<20}{pct:.1f}% ({estado})")

Modelo         Último checkpoint   % completado
gemma2_9b      12027               89.1% (en curso)


In [13]:
from huggingface_hub import login, list_repo_files

login()

REPO = "santiagortegasegura/emotion-futbol-checkpoints"

files = list_repo_files(REPO, repo_type="dataset")
print(f"Total de archivos encontrados: {len(files)}")

carpetas = sorted(set(f.split("/")[0] for f in files))
print("Carpetas de primer nivel encontradas:")
for c in carpetas:
    print(" -", c)

Total de archivos encontrados: 78
Carpetas de primer nivel encontradas:
 - .gitattributes
 - gemma2_2b
 - gemma2_9b
 - llama_3b
 - llama_8b
 - mistral_7b
 - phi35_mini
 - phi3_mini
 - qwen_1_5b
 - qwen_3b
 - qwen_7b


In [15]:
from huggingface_hub import login, list_repo_files
import re

login()

REPO = "santiagortegasegura/emotion-futbol-checkpoints"
TOTAL_STEPS = 13500

files = list_repo_files(REPO, repo_type="dataset")

modelos = {}
for f in files:
    m = re.match(r"([^/]+)/checkpoint-(\d+)/", f)
    if m:
        clave, paso = m.group(1), int(m.group(2))
        modelos[clave] = max(modelos.get(clave, 0), paso)

print(f"{'Modelo':<15}{'Último checkpoint':<20}{'% completado'}")
for clave, paso in sorted(modelos.items()):
    pct = paso / TOTAL_STEPS * 100
    estado = "COMPLETO" if paso >= TOTAL_STEPS else "en curso"
    print(f"{clave:<15}{paso:<20}{pct:.1f}% ({estado})")

Modelo         Último checkpoint   % completado
gemma2_9b      12027               89.1% (en curso)


In [17]:
from huggingface_hub import login, list_repo_files
import re

login()

REPO = "santiagortegasegura/emotion-futbol-checkpoints"
TOTAL_STEPS = 13500

files = list_repo_files(REPO, repo_type="dataset")

claves = ["qwen_1_5b","llama_3b","qwen_3b","phi35_mini","phi3_mini",
          "mistral_7b","qwen_7b","llama_8b","gemma2_2b","gemma2_9b"]

estado = {}
for clave in claves:
    completo = f"{clave}/ENTRENAMIENTO_COMPLETO.txt" in files
    pasos = [int(re.match(rf"{clave}/checkpoint-(\d+)/", f).group(1))
             for f in files if re.match(rf"{clave}/checkpoint-(\d+)/", f)]
    ultimo_paso = max(pasos) if pasos else None
    if completo:
        estado[clave] = ("COMPLETO", TOTAL_STEPS, 100.0)
    elif ultimo_paso is not None:
        estado[clave] = ("en curso", ultimo_paso, ultimo_paso / TOTAL_STEPS * 100)
    else:
        estado[clave] = ("sin iniciar", 0, 0.0)

print(f"{'Modelo':<15}{'Estado':<14}{'Último paso':<14}{'% completado'}")
for clave in claves:
    st, paso, pct = estado[clave]
    print(f"{clave:<15}{st:<14}{paso:<14}{pct:.1f}%")

Modelo         Estado        Último paso   % completado
qwen_1_5b      COMPLETO      13500         100.0%
llama_3b       COMPLETO      13500         100.0%
qwen_3b        COMPLETO      13500         100.0%
phi35_mini     COMPLETO      13500         100.0%
phi3_mini      COMPLETO      13500         100.0%
mistral_7b     COMPLETO      13500         100.0%
qwen_7b        COMPLETO      13500         100.0%
llama_8b       COMPLETO      13500         100.0%
gemma2_2b      COMPLETO      13500         100.0%
gemma2_9b      en curso      12027         89.1%
